# Alpamayo 1.5: Offline Local Navigation Comparison

This notebook runs offline local trajectory inference on the same scene window with two conditions:

- **no nav**
- **with nav**

It is designed to validate whether a natural-language navigation instruction changes the predicted trajectory.

Key features:
1. Automatically selects the **last valid t0** by default
2. Compares **no-nav** vs **with-nav** on the same window
3. Draws both predicted trajectories together with GT on the same BEV plot
4. Shows the current-timestep **Front_camera** image
5. Prints CoT for both conditions
6. Summarizes metrics for both conditions


In [ ]:
import os
import sys
import textwrap
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import av
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)

print('repo_root =', repo_root)
print('src_path =', src_path)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)

# ===== Paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Camera =====
TARGET_CAMERA_FILENAME = 'Front_camera.mp4'
FRONT_FRAME_T = -1

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)
EVAL_XY_ROTATION_DEG = -90.0

# ===== Window selection =====
AUTO_PICK_LAST_VALID_T0 = True
SINGLE_T0_SOD = 175500.23  # fallback when AUTO_PICK_LAST_VALID_T0=False
LAST_VALID_T0_OFFSET = 0  # 0 means the last one, 1 means the second last one

# ===== Navigation comparison =====
NAV_TEXT = 'Turn right at the upcoming intersection'
PRINT_FULL_COT = True
COT_MAX_CHARS_FOR_FIGURE = 320
COT_WRAP_WIDTH = 68

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('SEED =', SEED)
print('CLIP_DIR =', CLIP_DIR)
print('AUTO_PICK_LAST_VALID_T0 =', AUTO_PICK_LAST_VALID_T0)
print('NUM_TRAJ_SAMPLES =', NUM_TRAJ_SAMPLES)
print('NAV_TEXT =', NAV_TEXT)
print('TARGET_CAMERA_FILENAME =', TARGET_CAMERA_FILENAME)


### Helper functions


In [ ]:
def compute_max_valid_t0_range(cache, *, num_history_steps, num_future_steps, time_step, num_frames, fps, frame0_gps_time_sod):
    pose_min = float(cache.pose_times_sod[0])
    pose_max = float(cache.pose_times_sod[-1])

    history_left_span = (num_history_steps - 1) * time_step
    future_right_span = num_future_steps * time_step

    pose_t0_min = pose_min + history_left_span
    pose_t0_max = pose_max - future_right_span

    image_left_span = (num_frames - 1) * time_step
    image_t0_min = frame0_gps_time_sod + image_left_span
    return pose_t0_min, pose_t0_max, image_t0_min


def build_valid_t0_list(cache, *, num_history_steps, num_future_steps, time_step, num_frames, fps, frame0_gps_time_sod, step_sec=1.0):
    pose_t0_min, pose_t0_max, image_t0_min = compute_max_valid_t0_range(
        cache,
        num_history_steps=num_history_steps,
        num_future_steps=num_future_steps,
        time_step=time_step,
        num_frames=num_frames,
        fps=fps,
        frame0_gps_time_sod=frame0_gps_time_sod,
    )

    t0_min = max(pose_t0_min, image_t0_min)
    t0_max = pose_t0_max
    if t0_max < t0_min:
        raise ValueError(f'No valid t0 range. Computed range: [{t0_min:.3f}, {t0_max:.3f}]')

    return list(np.arange(t0_min, t0_max + 1e-9, step_sec, dtype=np.float64))


def pick_target_t0(cache) -> float:
    if not AUTO_PICK_LAST_VALID_T0:
        return float(SINGLE_T0_SOD)

    t0_list = build_valid_t0_list(
        cache,
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        step_sec=1.0,
    )

    if len(t0_list) == 0:
        raise ValueError('No valid t0 found.')

    idx = max(0, len(t0_list) - 1 - int(LAST_VALID_T0_OFFSET))
    return float(t0_list[idx])


def build_nav_conditions(nav_text: str):
    return {
        'no_nav': None,
        'with_nav': nav_text,
    }


def format_nav_label(nav_text: str | None) -> str:
    return '<none>' if nav_text is None else str(nav_text)


def truncate_text(s: str, max_chars: int) -> str:
    s = '' if s is None else str(s).strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + ' ...'


def wrap_text_block(s: str, width: int) -> str:
    s = '' if s is None else str(s).strip()
    return textwrap.fill(s, width=width)


def extract_metrics_row(result: dict) -> dict:
    df_metrics = result.get('df_metrics', None)
    if df_metrics is None or len(df_metrics) == 0:
        return {}
    return df_metrics.iloc[0].to_dict()


def decode_single_frame(video_path: Path, frame_index: int) -> np.ndarray:
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for idx, frame in enumerate(container.decode(stream)):
            if idx == frame_index:
                return frame.to_ndarray(format='rgb24')
    raise IndexError(f'Frame index {frame_index} out of range for {video_path}')


def pick_camera_row_by_exact_filename(clip_dir: Path, filename: str):
    camera_paths = helper.discover_offline_camera_files(clip_dir)
    camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
    matches = [(row, p) for row, p in enumerate(camera_paths_sorted) if p.name == filename]

    if len(matches) == 0:
        raise ValueError(f'No camera file matched exact filename={filename!r}')
    if len(matches) > 1:
        raise ValueError(f'Multiple camera files matched exact filename={filename!r}')

    row, path = matches[0]
    return row, path


def run_single_condition_from_loaded_data(data: dict, *, nav_text, model, processor):
    result = run_offline_inference_window(
        data=data,
        model=model,
        processor=processor,
        device=DEVICE,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        time_step=TIME_STEP,
        eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
        nav_text=nav_text,
    )
    result['metrics_row'] = extract_metrics_row(result)
    return result


def run_navigation_comparison_window(t0_sod, *, nav_text, model, processor, front_cam_row, front_video_path):
    data = load_offline_dataset(
        clip_dir=CLIP_DIR,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=False,
        image_size=IMAGE_SIZE,
    )

    conditions = build_nav_conditions(nav_text)
    outputs = {}
    for cond_name, cond_nav_text in conditions.items():
        outputs[cond_name] = run_single_condition_from_loaded_data(
            data,
            nav_text=cond_nav_text,
            model=model,
            processor=processor,
        )

    requested_front_frame_idx = int(data['actual_video_frame_indices'][front_cam_row, FRONT_FRAME_T].item())
    requested_front_ts = float(data['absolute_timestamps_sod'][front_cam_row, FRONT_FRAME_T].item())
    front_img = decode_single_frame(front_video_path, requested_front_frame_idx)

    return {
        't0_sod': float(t0_sod),
        'data': data,
        'nav_text': nav_text,
        'conditions': outputs,
        'front_img': front_img,
        'front_frame_idx': requested_front_frame_idx,
        'front_frame_ts': requested_front_ts,
    }


def plot_best_sample(ax, best_xyz, color, label):
    xy = best_xyz[:, :2]
    ax.plot(
        xy[:, 0], xy[:, 1], 'o-',
        color=color,
        linewidth=2.8,
        markersize=4.0,
        alpha=0.98,
        label=label,
    )


def render_navigation_comparison_figure(cmp_result):
    fig = plt.figure(figsize=(20, 7))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.25, 1.15])

    ax_img = fig.add_subplot(gs[0, 0])
    ax_bev = fig.add_subplot(gs[0, 1])
    ax_text = fig.add_subplot(gs[0, 2])

    no_nav = cmp_result['conditions']['no_nav']
    with_nav = cmp_result['conditions']['with_nav']

    gt_xyz = with_nav['gt_xyz_plot']
    hist_xyz = with_nav['hist_xyz_plot']

    xlim, ylim = _compute_adaptive_xy_limits(
        hist_xyz,
        gt_xyz,
        no_nav['pred_best_xyz'],
        with_nav['pred_best_xyz'],
        min_span=20.0,
        pad_ratio=0.12,
        pad_min=2.0,
    )

    ax_img.imshow(cmp_result['front_img'])
    ax_img.set_title(
        f"Front camera | frame={cmp_result['front_frame_idx']}\n"
        f"ts={cmp_result['front_frame_ts']:.3f}"
    )
    ax_img.axis('off')

    ax_bev.plot(hist_xyz[:, 0], hist_xyz[:, 1], 'b-o', linewidth=2.0, markersize=3.0, label='History')
    ax_bev.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=3.0, markersize=3.2, label='GT')

    plot_best_sample(ax_bev, no_nav['pred_best_xyz'], color='red', label='no nav pred')
    plot_best_sample(ax_bev, with_nav['pred_best_xyz'], color='blue', label='with nav pred')

    ax_bev.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
    ax_bev.set_xlabel('x')
    ax_bev.set_ylabel('y')
    ax_bev.set_title(
        f"No-nav vs nav trajectory | t0={cmp_result['t0_sod']:.3f}\n"
        f"nav={cmp_result['nav_text']}"
    )
    ax_bev.set_xlim(*xlim)
    ax_bev.set_ylim(*ylim)
    ax_bev.set_aspect('equal', adjustable='box')
    ax_bev.grid(True, alpha=0.3)
    ax_bev.legend(loc='best', fontsize=9)

    no_nav_cot = truncate_text(no_nav.get('cot_value', ''), COT_MAX_CHARS_FOR_FIGURE)
    with_nav_cot = truncate_text(with_nav.get('cot_value', ''), COT_MAX_CHARS_FOR_FIGURE)

    text_blocks = []
    text_blocks.append(f"t0_sod: {cmp_result['t0_sod']:.3f}")
    text_blocks.append(f"nav: {cmp_result['nav_text']}")
    text_blocks.append('')
    text_blocks.append('CoT (no nav):')
    text_blocks.append(wrap_text_block(no_nav_cot, COT_WRAP_WIDTH))
    text_blocks.append('')
    text_blocks.append('CoT (with nav):')
    text_blocks.append(wrap_text_block(with_nav_cot, COT_WRAP_WIDTH))

    ax_text.text(
        0.01, 0.99,
        '\n'.join(text_blocks),
        transform=ax_text.transAxes,
        va='top', ha='left', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.95),
    )
    ax_text.axis('off')
    ax_text.set_title('Navigation / CoT summary')

    plt.tight_layout()
    return fig


def print_all_cots(cmp_result):
    print('\n===== CoT (no nav) =====')
    print(cmp_result['conditions']['no_nav'].get('cot_value', ''))

    print('\n===== CoT (with nav) =====')
    print(cmp_result['conditions']['with_nav'].get('cot_value', ''))


def build_condition_summary_table(cmp_result):
    rows = []
    cond_to_nav = {
        'no_nav': None,
        'with_nav': cmp_result['nav_text'],
    }

    for cond_name, result in cmp_result['conditions'].items():
        row = {
            't0_sod': cmp_result['t0_sod'],
            'condition': cond_name,
            'nav_text': format_nav_label(cond_to_nav[cond_name]),
        }
        row.update(result.get('metrics_row', {}))
        rows.append(row)

    return pd.DataFrame(rows)


def build_best_traj_table(cmp_result):
    gt = cmp_result['conditions']['with_nav']['gt_xyz_plot']
    no_nav_best = cmp_result['conditions']['no_nav']['pred_best_xyz']
    nav_best = cmp_result['conditions']['with_nav']['pred_best_xyz']

    n = len(gt)
    return pd.DataFrame({
        'step': np.arange(n),
        't_sec': np.arange(n) * TIME_STEP,
        'gt_x': gt[:, 0],
        'gt_y': gt[:, 1],
        'no_nav_best_x': no_nav_best[:, 0],
        'no_nav_best_y': no_nav_best[:, 1],
        'nav_best_x': nav_best[:, 0],
        'nav_best_y': nav_best[:, 1],
    })


### Load cache, model, processor, and front camera path


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=False, force_rebuild=False)
front_cam_row, front_video_path = pick_camera_row_by_exact_filename(CLIP_DIR, TARGET_CAMERA_FILENAME)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Loaded cache, model, processor, and front camera path.')
print('front_cam_row =', front_cam_row)
print('front_video_path =', front_video_path)


### Pick target t0


In [ ]:
valid_t0_list = build_valid_t0_list(
    clip_cache,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    step_sec=1.0,
)

print('num valid t0 =', len(valid_t0_list))
print('first few valid t0 =', valid_t0_list[:5])
print('last few valid t0 =', valid_t0_list[-5:])

target_t0 = pick_target_t0(clip_cache)
print('target_t0 =', target_t0)


### Run no-nav vs with-nav comparison at the selected t0


In [ ]:
cmp_result = run_navigation_comparison_window(
    target_t0,
    nav_text=NAV_TEXT,
    model=model,
    processor=processor,
    front_cam_row=front_cam_row,
    front_video_path=front_video_path,
)

print('t0_sod =', cmp_result['t0_sod'])
print('nav_text =', cmp_result['nav_text'])
print('front_frame_idx =', cmp_result['front_frame_idx'])
print('front_frame_ts =', cmp_result['front_frame_ts'])

for cond_name, result in cmp_result['conditions'].items():
    metrics = result.get('metrics_row', {})
    print(
        f"{cond_name}: "
        f"best_sample_idx={metrics.get('best_sample_idx', 'N/A')}, "
        f"minADE_m={metrics.get('minADE_m', 'N/A')}, "
        f"final_point_error_m={metrics.get('final_point_error_m', 'N/A')}"
    )


### Compare no-nav vs with-nav trajectory and show front camera image


In [ ]:
fig = render_navigation_comparison_figure(cmp_result)
plt.show()
plt.close(fig)


### Print both CoTs


In [ ]:
if PRINT_FULL_COT:
    print_all_cots(cmp_result)
else:
    print('PRINT_FULL_COT = False, skipped full CoT printing.')


### Condition summary metrics


In [ ]:
df_summary = build_condition_summary_table(cmp_result)
display(df_summary)


### Best trajectory comparison table


In [ ]:
df_best_traj = build_best_traj_table(cmp_result)
display(df_best_traj)
